In [1]:
import os
from dotenv import load_dotenv

# Carica le variabili d'ambiente
load_dotenv()

# Importazioni necessarie
from datapizzai.clients import (
    ClientFactory, 
    OpenAIClient,
    AnthropicClient, 
    GoogleClient,
    MistralClient,
    AzureOpenAIClient
)
from datapizzai.clients.factory import Provider

## Custom Client

In [5]:
import os
import requests
from typing import Optional, Union, List
from pydantic import BaseModel

# Tipi DatapizzAI per compatibilità input/memory (opzionali)
from datapizzai.type import TextBlock
from datapizzai.memory import Memory
from dotenv import load_dotenv

# Carica le variabili d'ambiente
load_dotenv()


class SimpleResponse(BaseModel):
    """Struttura minima compatibile con l'uso nei nostri esempi."""
    text: str
    prompt_tokens_used: int = 0
    completion_tokens_used: int = 0
    stop_reason: str = "stop"


class HuggingFaceClient:
    """Adapter minimale per Hugging Face Inference API.

    Funziona con endpoint pubblici Inference API (task text-generation/text2text-generation).
    """

    def __init__(
        self,
        api_key: str,
        model: str,
        system_prompt: Optional[str] = None,
        temperature: float = 0.7,
    ):
        self.api_key = api_key
        self.model = model
        self.system_prompt = system_prompt or ""
        self.temperature = temperature
        self.endpoint = f"https://api-inference.huggingface.co/{self.model}"

    def _build_prompt(
        self,
        input: Optional[Union[str, List[TextBlock]]] = None,
        memory: Optional[Memory] = None,
    ) -> str:
        parts: List[str] = []
        if self.system_prompt:
            parts.append(self.system_prompt)
        if memory is not None:
            for turn in memory.memory:
                # Concatenazione semplice dei contenuti testuali presenti in memoria
                msg = " ".join(getattr(b, "content", "") for b in turn.blocks)
                parts.append(msg)
        if isinstance(input, str) and input:
            parts.append(input)
        elif isinstance(input, list) and input:
            parts.append(" ".join(b.content for b in input if isinstance(b, TextBlock)))
        return "\n\n".join(p for p in parts if p)

    def invoke(
        self,
        input: Optional[Union[str, List[TextBlock]]] = None,
        memory: Optional[Memory] = None,
    ) -> SimpleResponse:
        prompt = self._build_prompt(input=input, memory=memory)
        headers = {"Authorization": f"Bearer {self.api_key}"}
        payload = {
            "inputs": prompt,
            "parameters": {"temperature": self.temperature},
        }
        try:
            r = requests.post(self.endpoint, headers=headers, json=payload, timeout=60)
            r.raise_for_status()
            data = r.json()
            # Inference API ritorna una lista con 'generated_text' per i task di generazione
            if isinstance(data, list) and data and isinstance(data[0], dict):
                text = data[0].get("generated_text") or str(data[0])
            elif isinstance(data, dict) and "generated_text" in data:
                text = data["generated_text"]
            else:
                text = str(data)
        except Exception as e:
            text = f"Errore Hugging Face: {e}"
        return SimpleResponse(text=text)


# Test rapido del setup
if __name__ == "__main__":
    api_key = os.getenv("HUGGINGFACE_API_KEY")
    if not api_key:
        raise RuntimeError("Imposta HUGGINGFACE_API_KEY nel tuo .env")

    # Sostituisci con un modello disponibile su HF Inference API
    # Esempi: google/flan-t5-large (text2text), meta-llama/Llama-3.1-8B-Instruct (può richiedere accesso)
    client = HuggingFaceClient(
        api_key=api_key,
        model="google/gemma-3-270m",
        system_prompt="Sei un assistente AI utile e conciso.",
        temperature=0.7,
    )

    resp = client.invoke("Ciao! Presentati brevemente in due frasi.")
    print(f"Risposta: {resp.text}")

Risposta: Errore Hugging Face: 401 Client Error: Unauthorized for url: https://api-inference.huggingface.co/google/gemma-3-270m
